# Phục hồi Naive QA: chấm kết quả đã lưu

Không chạy lại generation, không gọi Ollama, không thay câu trả lời hay xóa ERROR.
Dùng đúng commit, fingerprint và cấu hình judge từ checkpoint. Các lượt lỗi generation vẫn được thống kê.

**Trước Run All:** Add Input Dataset `naive_qa_checkpoint`; bật Internet và Secret `GEMINI_API_KEY`. Không cần GPU, artifact gốc hoặc backend đang chạy. Hỗ trợ cả ZIP và thư mục `standard` đã được Kaggle giải nén.
Nếu hết quota, tải ZIP cuối notebook, đưa ZIP mới lên Input ở phiên sau và chọn nó bằng CHECKPOINT_ZIP; các lượt AI_JUDGED được giữ, không chấm lại.
Không tự đổi model/luân phiên key. Năm lỗi generation không được âm thầm chạy lại trên GPU khác rồi trộn vào run cũ.


In [ ]:
from pathlib import Path
import json, os, re, sys, subprocess, hashlib, zipfile, stat, shutil
REPO_URL = 'https://github.com/tvqui/legal-agentic-rag.git'
CHECKPOINT_ZIP = 'auto'  # hoặc đường dẫn ZIP / thư mục standard cụ thể trong /kaggle/input
GEMINI_SECRET_NAME = 'GEMINI_API_KEY'
MAX_NEW_JUDGMENTS = None  # None = chấm hết lượt OK chưa chấm; hoặc số dương để chia đợt
INPUT = Path('/kaggle/input')
RECOVERY = Path('/kaggle/working/naive_qa_recovery')
ROOT = RECOVERY/'code'
WORK = Path('/kaggle/working/unified_evaluation')
OUT = WORK/'labor120_v1/standard'
RUN = OUT/'naive_qa'
if not re.fullmatch(r'GEMINI_API_KEY(?:\.[1-9][0-9]*)?', GEMINI_SECRET_NAME):
    raise ValueError('Chọn tên Secret, không nhập giá trị API key.')
if MAX_NEW_JUDGMENTS is not None and (type(MAX_NEW_JUDGMENTS) is not int or MAX_NEW_JUDGMENTS < 1):
    raise ValueError('MAX_NEW_JUDGMENTS phải là số nguyên dương hoặc None.')


In [ ]:
# RESTORE_CHECKPOINT: chỉ đọc ZIP dữ liệu, không thực thi code trong ZIP.
from pathlib import PurePosixPath

def restore_checkpoint(archive, out):
    import shutil
    archive = Path(archive)
    if archive.is_symlink():
        raise ValueError('Không nhận checkpoint qua symlink.')
    required = ['settings.json','bundle120/manifest.json','bundle120/corpus.jsonl',
        'bundle120/questions.jsonl','naive_qa/run_manifest.json',
        'naive_qa/responses.jsonl','naive_qa/progress.json']
    directory = archive.is_dir()
    if directory:
        if out.resolve().is_relative_to(archive.resolve()):
            raise ValueError('Thư mục đích không được nằm trong thư mục nguồn.')
        members = sorted(archive.rglob('*'))
        if not members or len(members) > 10000:
            raise ValueError('Checkpoint rỗng hoặc vượt giới hạn số file.')
        entries = []
        total = 0
        seen = set()
        for member in members:
            relative = member.relative_to(archive).as_posix()
            parts = PurePosixPath(relative).parts
            if (member.is_symlink() or ':' in relative or chr(92) in relative
                or any(part in {'.env','.venv','cache','kaggle.json'} for part in parts)
                or relative.casefold() in seen):
                raise ValueError('Thư mục checkpoint chứa đường dẫn không an toàn.')
            seen.add(relative.casefold())
            if member.is_file():
                total += member.stat().st_size
                if total > 2_000_000_000:
                    raise ValueError('Checkpoint vượt giới hạn dung lượng.')
                entries.append([relative,hashlib.sha256(member.read_bytes()).hexdigest()])
            elif not member.is_dir():
                raise ValueError('Không nhận file đặc biệt trong checkpoint.')
        if any(not (archive/name).is_file() for name in required):
            raise ValueError('Thư mục thiếu file checkpoint bắt buộc.')
        # Định danh thư mục dựa trên tên và nội dung từng file, không dựa vào tên Dataset.
        digest = hashlib.sha256(json.dumps(entries,ensure_ascii=False,separators=(',',':')).encode('utf-8')).hexdigest()
    else:
        digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    marker = out/'.recovery_checkpoint.sha256'
    if out.exists():
        if not marker.is_file() or marker.read_text(encoding='utf-8').strip() != digest:
            raise RuntimeError('OUT đã có dữ liệu khác. Dùng phiên mới; không ghi đè checkpoint.')
        print('Giữ checkpoint đang phục hồi, không ghi đè kết quả đã chấm.', flush=True)
        return digest
    if directory:
        out.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(archive, out)
        marker.write_text(digest, encoding='utf-8')
        return digest
    with zipfile.ZipFile(archive) as z:
        infos = z.infolist()
        if not infos or len(infos) > 10000 or sum(i.file_size for i in infos) > 2_000_000_000:
            raise ValueError('ZIP rỗng hoặc vượt giới hạn checkpoint.')
        seen = set()
        for i in infos:
            name = i.filename
            parts = PurePosixPath(name).parts
            if (not parts or parts[0] != 'standard' or name.startswith('/')
                or '..' in parts or ':' in name or chr(92) in name
                or stat.S_ISLNK(i.external_attr >> 16)
                or any(p in {'.env', '.venv', 'cache', 'kaggle.json'} for p in parts)
                or name.casefold() in seen):
                raise ValueError('ZIP chứa đường dẫn không an toàn hoặc file không thuộc checkpoint.')
            seen.add(name.casefold())
        if z.testzip() is not None:
            raise ValueError('ZIP hỏng CRC.')
        required = ['settings.json','bundle120/manifest.json','bundle120/corpus.jsonl',
            'bundle120/questions.jsonl','naive_qa/run_manifest.json',
            'naive_qa/responses.jsonl','naive_qa/progress.json']
        if any('standard/'+n not in z.namelist() for n in required):
            raise ValueError('ZIP thiếu file checkpoint bắt buộc.')
        out.parent.mkdir(parents=True, exist_ok=True)
        z.extractall(out.parent)
    marker.write_text(digest, encoding='utf-8')
    return digest

def find_checkpoint(input_root, selected='auto'):
    if selected != 'auto':
        candidate = Path(selected)
        if not candidate.exists():
            raise FileNotFoundError('Không có checkpoint tại đường dẫn đã chọn.')
        return candidate
    candidates = list(Path(input_root).rglob('naive_qa_checkpoint.zip'))
    # Kaggle có thể đã giải nén ZIP khi tạo Dataset.
    for manifest in Path(input_root).rglob('run_manifest.json'):
        if manifest.parent.name != 'naive_qa':
            continue
        candidate = manifest.parent.parent
        if ((candidate/'settings.json').is_file()
            and (candidate/'bundle120/manifest.json').is_file()
            and (candidate/'naive_qa/responses.jsonl').is_file()):
            candidates.append(candidate)
    candidates = sorted(set(candidates))
    if len(candidates) != 1:
        print('Checkpoint tìm thấy:',[str(p) for p in candidates[:10]],flush=True)
        raise RuntimeError('Cần đúng một ZIP hoặc thư mục checkpoint; điền CHECKPOINT_ZIP bằng đường dẫn cụ thể nếu có nhiều bộ.')
    return candidates[0]

ARCHIVE = find_checkpoint(INPUT, CHECKPOINT_ZIP)
print('Checkpoint nguồn:',ARCHIVE, '; dạng:', 'thư mục đã giải nén' if ARCHIVE.is_dir() else 'ZIP',flush=True)
CHECKPOINT_SHA256 = restore_checkpoint(ARCHIVE, OUT)
M = json.loads((RUN/'run_manifest.json').read_text(encoding='utf-8'))
BUNDLE = OUT/'bundle120'
BM = json.loads((BUNDLE/'manifest.json').read_text(encoding='utf-8'))
for filename, key in [('corpus.jsonl','corpus_jsonl_sha256'),('questions.jsonl','questions_sha256')]:
    if hashlib.sha256((BUNDLE/filename).read_bytes()).hexdigest() != BM[key]:
        raise ValueError('Fingerprint bundle không khớp: '+filename)
if M['task'] != 'qa' or M['methods'] != ['naive_bm25','naive_dense','naive_hybrid']:
    raise ValueError('Checkpoint không phải ba baseline Naive QA dự kiến.')
if Path(M['settings']['bundle_corpus']) != BUNDLE/'corpus.jsonl':
    raise ValueError('Đường dẫn bundle gốc khác notebook; không sửa manifest để bỏ kiểm tra.')
RECORDS = [json.loads(s) for s in (RUN/'responses.jsonl').read_text(encoding='utf-8').splitlines() if s.strip()]
expected = {(qid, method, repeat) for qid in M['selected_ids'] for method in M['methods']
            for repeat in range(1, M['repeats']+1)}
actual = [(r['id'],r['method'],r['repeat']) for r in RECORDS]
if len(actual) != len(set(actual)) or set(actual) != expected:
    raise ValueError('Thiếu, trùng hoặc thừa bản ghi so với lịch đo đóng băng.')
COMMIT = M['environment']['git_commit']
if not re.fullmatch(r'[0-9a-f]{40}', COMMIT) or M['environment'].get('git_tracked_dirty') is not False:
    raise ValueError('Không xác minh được commit sạch của phép đo gốc.')
JUDGE_MODEL = M['settings']['judge']['model']
if M['settings']['judge']['provider'] != 'gemini':
    raise ValueError('Notebook này chỉ phục hồi judge Gemini đã đóng băng.')
print('Đã phục hồi',len(RECORDS),'lượt; OK:',sum(r['status']=='OK' for r in RECORDS),
      '; ERROR:',sum(r['status']=='ERROR' for r in RECORDS),'; judge:',JUDGE_MODEL,flush=True)


In [ ]:
# Lấy đúng mã nguồn gốc; KHÔNG checkout origin/main mới nhất.
RECOVERY.mkdir(parents=True, exist_ok=True)
def command(arguments, **kw):
    return subprocess.run(list(map(str,arguments)),check=True,**kw)
if not ROOT.exists():
    command(['git','clone','--no-checkout',REPO_URL,ROOT])
else:
    dirty = subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=ROOT,text=True)
    if dirty.strip():
        raise RuntimeError('Code phục hồi bị sửa; không ghi đè hoặc bỏ kiểm fingerprint.')
command(['git','checkout','--detach',COMMIT],cwd=ROOT)
for relative, digest in M['source_tree'].items():
    parts = PurePosixPath(relative).parts
    if not parts or '..' in parts or ':' in relative or chr(92) in relative or relative.startswith('/'):
        raise ValueError('Đường dẫn source_tree không hợp lệ.')
    p = ROOT/relative
    if not p.is_file() or hashlib.sha256(p.read_bytes()).hexdigest() != digest:
        raise ValueError('Code không khớp phép đo gốc: '+relative)
print('Khớp',len(M['source_tree']),'file mã nguồn; commit',COMMIT,flush=True)
VENV = RECOVERY/'.venv'
command([sys.executable,'-m','venv','--system-site-packages','--without-pip',VENV])
PYTHON = VENV/'bin/python'
pins = []
for name in ('numpy','psutil','google-genai','matplotlib'):
    version = M['environment']['packages'].get(name)
    if not isinstance(version,str) or not re.fullmatch(r'[0-9][A-Za-z0-9.+_-]*', version):
        raise ValueError('Thiếu phiên bản thư viện gốc: '+name)
    pins.append(name+'=='+version)
command([sys.executable,'-m','pip','--python',PYTHON,'install','-q','-e',str(ROOT)+'[online]',*pins])
# Đối chiếu toàn bộ source_tree bằng chính bộ đo trước khi nạp key hoặc gọi API.
probe = "import sys,json;sys.path.insert(0,'đo');import eval_core as e;m=e.read(sys.argv[1]);assert e.source_tree()==m['source_tree'];e.load_bundle(sys.argv[2]);e.dump(sys.argv[3],e.environment());print('CODE_AND_BUNDLE_VERIFIED')"
command([PYTHON,'-X','utf8','-c',probe,RUN/'run_manifest.json',BUNDLE,
    OUT/'posthoc_environment.json'],cwd=ROOT)


In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ.pop('GEMINI_API_KEY',None)
try:
    key = UserSecretsClient().get_secret(GEMINI_SECRET_NAME)
except Exception:
    raise RuntimeError('Bật quyền truy cập Secret '+GEMINI_SECRET_NAME) from None
if not isinstance(key,str) or not key.strip():
    raise RuntimeError('Secret đã chọn trống.')
os.environ['GEMINI_API_KEY'] = key.strip()
os.environ['GEMINI_MODEL'] = JUDGE_MODEL
os.environ['VN_LABOR_GEMINI_RPM'] = '13'
# Không in key, không gọi probe dư, không dùng backend hoặc Ollama.
print('Đã nạp Secret; giữ judge model:',JUDGE_MODEL,flush=True)


In [ ]:
# JUDGE_SAVED_ONLY: runner bỏ qua ERROR generation và AI_JUDGED đã có.
from IPython.display import FileLink, display, Markdown
state = {'action':'JUDGE_SAVED_ONLY','checkpoint_sha256':CHECKPOINT_SHA256,
    'generation_rerun':False,'generation_errors_retained':sum(r['status']=='ERROR' for r in RECORDS),
    'commit':COMMIT,'judge_model':JUDGE_MODEL}
try:
    args = [PYTHON,'-u','đo/evaluate_baselines.py','judge','--run',RUN,'--allow-llm']
    if MAX_NEW_JUDGMENTS is not None:
        args += ['--max-new-judgments',str(MAX_NEW_JUDGMENTS)]
    result = subprocess.run(list(map(str,args)),cwd=ROOT,check=False)
    state['judge_exit'] = result.returncode
    if result.returncode not in (0,2):
        raise subprocess.CalledProcessError(result.returncode,result.args)
    progress_file = RUN/'judge_progress.json'
    if progress_file.exists():
        state['judge_progress'] = json.loads(progress_file.read_text(encoding='utf-8'))
        print('Tiến độ judge:',state['judge_progress'],flush=True)
    if result.returncode == 2:
        print('Judge còn lỗi hoặc bị tạm dừng; điểm đã chấm vẫn được giữ. Không coi là hoàn tất.',flush=True)
except Exception as exc:
    state['exception_type'] = type(exc).__name__
    raise
finally:
    # Chỉ thêm điểm judge; kiểm tra các câu trả lời và kết quả generation không bị đổi.
    try:
        after = [json.loads(s) for s in (RUN/'responses.jsonl').read_text(encoding='utf-8').splitlines() if s.strip()]
        def generation_only(r):
            return {k:v for k,v in r.items() if k not in {'judge','judge_ms','metrics'}}
        if [generation_only(r) for r in after] != [generation_only(r) for r in RECORDS]:
            raise RuntimeError('Generation bị thay đổi; không dùng ZIP này như kết quả gốc.')
        state['generation_unchanged'] = True
    except Exception as exc:
        state['generation_unchanged'] = False
        state['verification_error_type'] = type(exc).__name__
        print('CẢNH BÁO: kiểm tra tính toàn vẹn generation không đạt.',flush=True)
    (OUT/'recovery_status.json').write_text(json.dumps(state,ensure_ascii=False,indent=2),encoding='utf-8')
    archive = Path(shutil.make_archive(str(OUT)+'_judge_recovery','zip',root_dir=OUT.parent,base_dir=OUT.name))
    print('TẢI FILE:',archive,flush=True)
    display(FileLink(str(archive)))
    table = RUN/'report/comparison.md'
    if table.exists():
        display(Markdown(table.read_text(encoding='utf-8')))


## Đọc kết quả

Tải **standard_judge_recovery.zip** ở Output. ZIP vẫn có gốc `standard/`, dùng được làm checkpoint lần chấm tiếp.
`judge_progress.complete_for_saved_answers` chỉ nói các câu OK đã được xử lý chấm; xem AI_JUDGED, N/A và ERROR riêng.
Năm lỗi generation không biến thành điểm 0 hoặc câu trả lời thành công. Gold vẫn DRAFT; AI judge không phải xác nhận pháp lý.

**Chạy bổ sung lỗi generation:** run gốc dùng EXACT_DEVICE và khóa phiên bản Ollama/model/dependencies. Nếu mất phiên GPU cũ, không giả UUID hoặc sửa comparison_key để resume. Cần một thí nghiệm bổ sung riêng với provenance rõ ràng hoặc đo lại cả bộ trên môi trường mới.
